# 🔁 Cross-Validation & Resampling
**ISLP Chapter 5 · Pattern Recognition for the Rest of Us**

> How do you honestly estimate how well your model will perform on new data — without collecting new data? Cross-validation is the answer, and it's one of the most practically important techniques in this entire course.

---
### What you'll learn
- Why a simple train/test split can be misleading
- Leave-one-out cross-validation (LOOCV) — the extreme case
- K-fold cross-validation — the standard in practice
- The bootstrap — estimating uncertainty in any statistic
- How to use CV for model selection

### Time
~55 minutes

## 🎯 Part 1 — The Problem with a Single Train/Test Split

When we split data randomly into training and test sets, the test error estimate depends on *which observations happened to land in the test set*. With small datasets this variance can be huge — you might get lucky or unlucky.

**Example:** With 100 observations and a 70/30 split, there are C(100,30) ≈ 3×10²⁵ possible test sets. Each gives a different error estimate.

Cross-validation solves this by systematically using *all* the data for both training and validation, averaging the results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (KFold, LeaveOneOut, cross_val_score,
                                      train_test_split)
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.datasets import load_iris, load_diabetes

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})

# Demonstrate variance of a single train/test split
np.random.seed(0)
n = 80
X = np.random.uniform(0,1,n)
Y = 2*X + np.random.normal(0, 0.5, n)
X2d = X.reshape(-1,1)

# Repeat the split 50 times and collect test MSE
mse_list = []
for seed in range(50):
    X_tr, X_te, Y_tr, Y_te = train_test_split(X2d, Y, test_size=0.3, random_state=seed)
    lr = LinearRegression().fit(X_tr, Y_tr)
    mse_list.append(mean_squared_error(Y_te, lr.predict(X_te)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(50), mse_list, color='#1e5fa8', alpha=0.7, edgecolor='white')
ax.axhline(np.mean(mse_list), color='#e85d20', lw=2, ls='--', label=f'Mean = {np.mean(mse_list):.3f}')
ax.set_xlabel('Random seed (different train/test split)')
ax.set_ylabel('Test MSE')
ax.set_title('Same model, 50 different splits — test MSE varies widely')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nTest MSE range: {min(mse_list):.3f} — {max(mse_list):.3f}")
print(f"Standard deviation: {np.std(mse_list):.3f}")
print("📌 Which split should you trust? Cross-validation averages them all.")

## 🔂 Part 2 — K-Fold Cross-Validation

**K-fold CV** divides the data into K equal-sized groups (folds). For each fold k:
1. Train on the other K-1 folds
2. Validate on fold k
3. Record the error

After K iterations, average the K error estimates.

```
CV(K) = (1/K) × Σ MSEₖ
```

**Typical values:** K=5 or K=10. K=10 is the most common in practice.

**Why not K=2?** High bias — each training set is only half the data.  
**Why not K=n (LOOCV)?** Very slow, high variance in some settings.

In [ ]:
# Visualize how K-fold CV works
fig, ax = plt.subplots(figsize=(11, 4))
n_obs, K = 30, 5
fold_size = n_obs // K

colors = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1','#0e7490']

for k in range(K):
    for i in range(n_obs):
        fold = i // fold_size
        if fold == k:
            color = colors[k]
            label = 'Validation' if i == k * fold_size else ''
        else:
            color = '#d0d8e8'
            label = ''
        ax.barh(k, 1, left=i, color=color, edgecolor='white', height=0.7)

ax.set_yticks(range(K))
ax.set_yticklabels([f'Fold {k+1}' for k in range(K)])
ax.set_xlabel('Observation index')
ax.set_title('5-Fold Cross-Validation — colored = validation fold, gray = training')

# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=f'Fold {i+1} as validation') for i in range(K)]
legend_elements.append(Patch(facecolor='#d0d8e8', label='Training'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# K-fold CV in practice — model selection with sklearn
# Dataset: Auto (car fuel efficiency)
url = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Auto.csv'
auto = pd.read_csv(url, na_values='?').dropna()

X_auto = auto[['horsepower']].values
y_auto = auto['mpg'].values

print(f"Auto dataset: {auto.shape[0]} cars")
print(f"Predicting: mpg  |  Predictor: horsepower\n")

# Compare polynomial degrees 1-8 using 10-fold CV
kf = KFold(n_splits=10, shuffle=True, random_state=42)
degrees = range(1, 9)
cv_scores = []

for d in degrees:
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    scores = cross_val_score(pipe, X_auto, y_auto, cv=kf,
                             scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())

best_d = degrees[np.argmin(cv_scores)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(degrees), cv_scores, 'o-', color='#1e5fa8', lw=2, markersize=7)
ax.axvline(best_d, color='#e85d20', lw=1.5, ls='--',
           label=f'Best: degree {best_d} (CV MSE={cv_scores[best_d-1]:.2f})')
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('10-Fold CV MSE')
ax.set_title('Model Selection via Cross-Validation (Auto dataset)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 CV selects degree {best_d} as optimal — degree 1 underfits, higher degrees overfit.")

## 🔂 Part 3 — Leave-One-Out CV (LOOCV)

LOOCV is K-fold CV where K = n. Each observation gets its own validation fold.

**Pros:** Nearly unbiased estimate of test error (trained on n-1 observations ≈ full data).  
**Cons:** Must fit the model n times — expensive for large n. Can have high variance.

**When to use LOOCV:** Small datasets (n < 100) where you can't afford to leave much data out.

In [ ]:
# LOOCV vs 5-fold vs 10-fold — compare estimates and runtime
import time

results = {}
for name, cv in [('LOOCV', LeaveOneOut()),
                 ('5-Fold CV', KFold(5, shuffle=True, random_state=42)),
                 ('10-Fold CV', KFold(10, shuffle=True, random_state=42))]:
    pipe = Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())])
    t0 = time.time()
    scores = cross_val_score(pipe, X_auto, y_auto, cv=cv,
                             scoring='neg_mean_squared_error')
    elapsed = time.time() - t0
    results[name] = {'mse': -scores.mean(), 'std': scores.std(), 'time': elapsed}

print("=" * 55)
print(f"{'Method':<15} {'CV MSE':>10} {'Std':>10} {'Time (s)':>10}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<15} {r['mse']:>10.3f} {r['std']:>10.3f} {r['time']:>10.3f}")
print("=" * 55)
print("\n📌 All three give similar MSE estimates.")
print("   LOOCV is slower and has higher std — 10-fold is the standard choice.")

## 🥾 Part 4 — The Bootstrap

The bootstrap estimates the variability of *any* statistic by resampling with replacement from your dataset.

**How it works:**
1. Sample n observations with replacement → bootstrap dataset B*
2. Compute your statistic on B* → get θ̂*
3. Repeat B times (usually B=1000)
4. The standard deviation of {θ̂₁*, θ̂₂*, ..., θ̂ᴮ*} estimates the SE of your statistic

**Key insight:** "with replacement" means some observations appear multiple times, others not at all (~37% are never sampled). Each bootstrap dataset is a plausible alternative sample from the population.

In [ ]:
# Bootstrap example: estimate standard error of the median
np.random.seed(42)
# Simulate a skewed distribution (e.g., income)
population_sample = np.concatenate([
    np.random.exponential(scale=50000, size=200),
    np.random.normal(200000, 30000, 20)   # a few high earners
])
population_sample = population_sample[population_sample > 0]
n = len(population_sample)

# Bootstrap
B = 2000
bootstrap_medians = []
for _ in range(B):
    boot_sample = np.random.choice(population_sample, size=n, replace=True)
    bootstrap_medians.append(np.median(boot_sample))

se_bootstrap = np.std(bootstrap_medians)
observed_median = np.median(population_sample)
ci_lower, ci_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: data distribution
axes[0].hist(population_sample, bins=40, color='#1e5fa8', alpha=0.7, edgecolor='white')
axes[0].axvline(observed_median, color='#e85d20', lw=2, label=f'Median = ${observed_median:,.0f}')
axes[0].set_title('Income Distribution (n=220)')
axes[0].set_xlabel('Income ($)')
axes[0].legend()

# Right: bootstrap distribution of median
axes[1].hist(bootstrap_medians, bins=50, color='#1a7a45', alpha=0.7, edgecolor='white')
axes[1].axvline(observed_median, color='#e85d20', lw=2, label=f'Observed median')
axes[1].axvline(ci_lower, color='#888', lw=1.5, ls='--')
axes[1].axvline(ci_upper, color='#888', lw=1.5, ls='--', label=f'95% CI: [${ci_lower:,.0f}, ${ci_upper:,.0f}]')
axes[1].set_title(f'Bootstrap Distribution of Median (B={B})')
axes[1].set_xlabel('Bootstrap Median ($)')
axes[1].legend(fontsize=9)

plt.suptitle('Bootstrap — Estimating Uncertainty in the Median', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nObserved median:    ${observed_median:,.0f}")
print(f"Bootstrap SE:       ${se_bootstrap:,.0f}")
print(f"95% CI:             [${ci_lower:,.0f}, ${ci_upper:,.0f}]")
print("\n📌 We never collected new data — the bootstrap created 2000 synthetic datasets from the one we have.")

In [ ]:
# CV for classification — Iris dataset
from sklearn.neighbors import KNeighborsClassifier
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

k_values = range(1, 31)
cv_accs = []

kf = KFold(n_splits=10, shuffle=True, random_state=42)
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_iris, y_iris, cv=kf, scoring='accuracy')
    cv_accs.append(scores.mean())

best_k = k_values[np.argmax(cv_accs)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_values), cv_accs, 'o-', color='#6b46c1', lw=2, markersize=5)
ax.axvline(best_k, color='#e85d20', lw=1.5, ls='--', label=f'Best k={best_k} (acc={max(cv_accs):.3f})')
ax.set_xlabel('Number of Neighbors (K)')
ax.set_ylabel('10-Fold CV Accuracy')
ax.set_title('Choosing K in KNN via Cross-Validation (Iris dataset)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 CV selects k={best_k} as optimal. Very small k overfits; very large k underfits.")

## 💼 Exercise

**Task 1:** Load the Auto dataset. Use 10-fold CV to compare Linear, Ridge (alpha=1), and Lasso (alpha=1) regression predicting `mpg` from all numeric features. Which has the best CV MSE?

**Task 2:** On a dataset of your choice, compare the CV estimates from 5-fold, 10-fold, and LOOCV. How different are they? At what n does LOOCV become impractical?

**Task 3:** Use the bootstrap to estimate the standard error of the *mean* horsepower in the Auto dataset. How does your bootstrap SE compare to the analytical SE (std/√n)?

In [ ]:
# Exercise workspace
url = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Auto.csv'
auto = pd.read_csv(url, na_values='?').dropna()

# Task 1: compare models with 10-fold CV
# YOUR CODE HERE

# Task 2: 5-fold vs 10-fold vs LOOCV
# YOUR CODE HERE

# Task 3: bootstrap SE of mean horsepower
# YOUR CODE HERE

In [ ]:
answers = {
    "q1": "",  # What does K-fold CV do that a single train/test split does not?
    "q2": "",  # Why is LOOCV sometimes worse than 10-fold CV despite being less biased?
    "q3": "",  # In bootstrap resampling, approximately what fraction of observations are never sampled?
    "q4": "",  # You have 20 observations. Which CV method would you choose and why?
    "q5": "",  # How does CV help with model selection? Give a specific example.
}
missing = [k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ All answered! Save: File → Save a copy in GitHub")

## 📚 Further Reading
| Resource | What it covers |
|----------|---------------|
| [ISLP Ch 5](https://intro-stat-learning.github.io/ISLP/) | Full CV and bootstrap treatment |
| [sklearn CV docs](https://scikit-learn.org/stable/modules/cross_validation.html) | All CV variants in sklearn |
| [Bias-Variance notebook](bias-variance.ipynb) | Why we need CV in the first place |
| [Regularization notebook](regularization.ipynb) | Uses CV to select λ |

---
*Pattern Recognition for the Rest of Us · ISLP Ch 5 · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*